# Create Customer Order Summary
1. Join the tables silver_customers, silver_orders, silver_addresses
2. Retrieve the latest Address of the customer
3. Calculate the following rules
    - total _orders
    - total_items_ordered
    - total_order_amopunt 

![image_1781071948428.png](./image_1781071948428.png "image_1781071948428.png")

![image_1781072016763.png](./image_1781072016763.png "image_1781072016763.png")

In [0]:
from pyspark.sql.functions import *

@dlt.materialized_view(
    name='gold_customer_order_summary',
    table_properties={'quality':'gold'},
    comment="Materialised view customer order summary"
)
def create_customer_order_summary():
    silver_customers=spark.read.table('LIVE.silver_customers') # Here we will not use spark.readStream
    silver_addresses=spark.read.table('LIVE.silver_addresses')
    silver_orders=spark.read.table('LIVE.silver_orders')
    customer_order_summary_df=(silver_customers.join(silver_addresses, (col('silver_customers.customer_id')==col('silver_addresses.customer_id'))  & 
                                                    (col('silver_addresses.__END_AT').isNull()))
    .join(silver_orders, col("silver_orders.customer_id"))
    .drop(col('silver_addresses.customer_id'), col('silver_orders.customer_id'))
    )
    return(
        customer_order_summary_df
            .groupBy(
                col('customer_id'),
                col('customer_name'),
                col('date_of_birth'),
                col('telephone'),
                col('email'),
                col('address_line_1'),
                col('city'),
                col('state'),
                col('postcode'),
            )
            .agg(
                countDistinct(col('order_id')).alias('total_orders'),
                sum(col('item_quantity')).alias('total_items_ordered'),
                sum(col('item_price')*col('item_quantity')).alias('ordered_amount')
                )
            )
    